# Pillow: working with image pixels

A small collection of images needs consistent thumbnails and a contact sheet. The challenge is to preserve useful content while changing dimensions, modes and file formats. Inspect aspect ratio, transparency and pixel values before deciding which transformation is suitable.

Pillow opens, transforms, draws, and saves raster images. We will create a simple geometric scene, inspect its pixels, compare resizing methods, work with transparency, and export an image-processing report.

Every source image is generated in this notebook. No downloads or external image files are needed. Run cells in order; outputs are written only under `outputs`.

## 1. Install and import

If needed, run `%pip install "pillow>=10" numpy ipython` in a separate cell. The package is named Pillow but imported as `PIL`. We use `display` for notebook previews rather than launching a separate image viewer.

In [ ]:
from pathlib import Path
import io
import json
import numpy as np
import PIL
from PIL import Image, ImageDraw, ImageFont, ImageOps, ImageFilter, ImageEnhance, ImageStat
from IPython.display import display

print("Pillow:", PIL.__version__)
output_dir = Path.cwd() / "outputs"
output_dir.mkdir(exist_ok=True)

## 2. Create a source image

An RGB pixel has red, green, and blue channels. Coordinates start at the upper-left corner: x increases to the right, y increases downward. Image sizes are `(width, height)`.

Drawing methods modify the target image. We create a 480 by 320 scene with simple shapes so every later transformation is reproducible.

In [ ]:
source = Image.new("RGB", (480, 320), "#d9edf7")
draw = ImageDraw.Draw(source)
draw.rectangle((0, 220, 479, 319), fill="#7ab87a")
draw.ellipse((350, 25, 430, 105), fill="#ffcf55")
draw.rectangle((120, 140, 280, 270), fill="#f4c095")
draw.polygon([(100, 145), (200, 65), (300, 145)], fill="#9b4b48")
draw.rectangle((180, 200, 220, 270), fill="#654b3a")
draw.rectangle((135, 165, 165, 195), fill="white")
source_path = output_dir / "pillow_source.png"
source.save(source_path)
display(source)

## 3. Open files and inspect metadata

`Image.open` initially reads enough to identify the file, then decodes pixels when required. Use a context manager and copy the loaded image when it must outlive the open file.

`ImageOps.exif_transpose` applies a camera's orientation metadata when present. Our generated PNG has no orientation tag, but this is a useful input step for photographs. Mode and file format are different concepts.

In [ ]:
with Image.open(source_path) as opened:
    print("Format:", opened.format, "Mode:", opened.mode, "Size:", opened.size)
    loaded = ImageOps.exif_transpose(opened).convert("RGB")
    loaded.load()
print("Corner pixel:", loaded.getpixel((0, 0)))
print("Bands:", loaded.getbands())

## 4. Compare image modes

| Mode | Channels | Typical use |
| --- | --- | --- |
| RGB | Red, green, blue | Opaque color images |
| RGBA | RGB plus alpha | Color with transparency |
| L | One 8-bit grayscale channel | Grayscale images and masks |
| P | Palette indices | Indexed-color images |

Conversion changes pixel representation. Converting to grayscale loses color information; converting back to RGB cannot recover it.

In [ ]:
grayscale = loaded.convert("L")
rgba = loaded.convert("RGBA")
print("Grayscale pixel:", grayscale.getpixel((0, 0)))
print("RGBA pixel:", rgba.getpixel((0, 0)))
display(grayscale)

## 5. Make a thumbnail without distorting its subject

Predict the output dimensions for resize and thumbnail. Compare the aspect ratio with the source. A requested bounding box does not necessarily mean that both output dimensions should exactly match it.

`resize` returns exactly the requested dimensions, which can distort proportions. `thumbnail` preserves the aspect ratio within a bounding box and modifies the image in place. Work on a copy when keeping the original.

LANCZOS is useful for smooth downsampling. NEAREST keeps hard pixel boundaries and is often appropriate for pixel art or categorical masks.

In [ ]:
stretched = loaded.resize((180, 180), resample=Image.Resampling.LANCZOS)
thumbnail = loaded.copy()
thumbnail.thumbnail((180, 180), resample=Image.Resampling.LANCZOS)
print("Original:", loaded.size, "Resized:", stretched.size, "Thumbnail:", thumbnail.size)
display(stretched, thumbnail)

## 6. Choose whether to crop, pad or leave empty space

Decide which matters more for the use case: retaining all content or filling every pixel of a fixed box. Compare fit, contain and pad with that decision in mind. Name any content lost to cropping.

`contain` fits the whole image within a box. `fit` fills the box by resizing and cropping. `pad` keeps the complete image and fills unused space with a background color.

All three preserve aspect ratio, but only fit and pad guarantee this example's exact square output dimensions.

In [ ]:
contained = ImageOps.contain(loaded, (200, 200), method=Image.Resampling.LANCZOS)
fitted = ImageOps.fit(loaded, (200, 200), method=Image.Resampling.LANCZOS)
padded = ImageOps.pad(loaded, (200, 200), method=Image.Resampling.LANCZOS, color="white")
print("Contain:", contained.size, "Fit:", fitted.size, "Pad:", padded.size)
display(contained, fitted, padded)

## 7. Crop a region

A crop box is `(left, upper, right, lower)`. The right and lower boundaries are excluded: this box has width 200 and height 230. Cropping outside an image can add pixels rather than reject the coordinates, so validate bounds when that behavior is unwanted.

In [ ]:
crop_box = (100, 50, 300, 280)
left, upper, right, lower = crop_box
if not (0 <= left < right <= loaded.width and 0 <= upper < lower <= loaded.height):
    raise ValueError("Crop box must be inside the source")
cropped = loaded.crop(crop_box)
print("Crop size:", cropped.size)
display(cropped)

## 8. Rotate and mirror

A positive rotation angle rotates counterclockwise. `expand=True` enlarges the output canvas so the rotated image is not clipped. Fill color controls newly exposed pixels.

For exact quarter turns, `transpose` avoids the interpolation needed for arbitrary angles.

In [ ]:
rotated = loaded.rotate(20, resample=Image.Resampling.BICUBIC, expand=True, fillcolor="white")
quarter_turn = loaded.transpose(Image.Transpose.ROTATE_90)
mirrored = ImageOps.mirror(loaded)
print("Quarter-turn size:", quarter_turn.size)
display(rotated, mirrored)

## 9. Check how transparency combines with the background

Predict where the source should remain visible through the overlay. Inspect edges and partially transparent pixels, not only the solid center. Transparency is part of the image data, not just a display decoration.

Alpha values range from 0 (transparent) to 255 (opaque). Draw a semitransparent badge on its own RGBA layer, then alpha-composite it over the scene.

The module-level `alpha_composite` requires matching sizes and compatible modes. Drawing on a separate layer makes opacity explicit.

In [ ]:
overlay = Image.new("RGBA", loaded.size, (0, 0, 0, 0))
overlay_draw = ImageDraw.Draw(overlay)
overlay_draw.rounded_rectangle((15, 15, 165, 55), radius=10, fill=(20, 40, 70, 190))
font = ImageFont.load_default()
overlay_draw.text((28, 29), "Pillow practice", fill="white", font=font)
composited = Image.alpha_composite(loaded.convert("RGBA"), overlay)
display(composited)

## 10. Use a mask to select pixels

A grayscale mask controls blending: white selects the first image, black selects the second, and intermediate values blend them. Here a circular mask reveals color inside a grayscale scene.

Blurring the mask creates a soft transition rather than a hard edge. This is different from blurring the scene itself.

In [ ]:
mask = Image.new("L", loaded.size, 0)
ImageDraw.Draw(mask).ellipse((80, 30, 320, 270), fill=255)
soft_mask = mask.filter(ImageFilter.GaussianBlur(radius=8))
spotlight = Image.composite(loaded, grayscale.convert("RGB"), soft_mask)
display(soft_mask, spotlight)

## 11. Apply filters and enhancement

Gaussian blur smooths local detail. Enhancement changes properties such as contrast or brightness. A factor of 1 retains the original setting; values below and above it decrease and increase the effect.

These operations change the image, so compare against an original rather than assuming that stronger enhancement is always better.

In [ ]:
blurred = loaded.filter(ImageFilter.GaussianBlur(radius=3))
high_contrast = ImageEnhance.Contrast(loaded).enhance(1.5)
brightened = ImageEnhance.Brightness(loaded).enhance(1.15)
display(blurred, high_contrast, brightened)

## 12. Bridge Pillow and NumPy

Pillow uses `(width, height)` for sizes, while an RGB NumPy array has shape `(height, width, channels)`. Array indexing is `[y, x]`.

Use a copy for editable pixels. Convert to a wider integer type before adding channel values so 8-bit arithmetic does not wrap around. Clip to 0–255 before converting back.

In [ ]:
pixels = np.array(loaded)
print("Array shape:", pixels.shape, "dtype:", pixels.dtype)
print("Same corner:", tuple(pixels[0, 0]), loaded.getpixel((0, 0)))
warmed_pixels = pixels.astype(np.int16)
warmed_pixels[:, :, 0] += 25
warmed_pixels = np.clip(warmed_pixels, 0, 255).astype(np.uint8)
warmed = Image.fromarray(warmed_pixels)
display(warmed)

## 13. Measure image values

ImageStat summarizes channel values. A grayscale histogram counts pixels at each of 256 intensity levels. Its counts sum to the number of pixels, not the number of color channels.

A histogram describes value frequencies but does not retain where pixels occur. Two very different arrangements can have the same histogram.

In [ ]:
channel_means = ImageStat.Stat(loaded).mean
histogram = grayscale.histogram()
print("Mean RGB channels:", [round(value, 2) for value in channel_means])
print("Histogram bins:", len(histogram))
print("Pixels counted:", sum(histogram))

## 14. Choose an export that preserves the intended appearance

Before saving, identify whether the image has transparency and whether the target format supports it. After reopening the file, compare mode, dimensions and appearance. A visually acceptable export may still have discarded alpha or changed pixel values.

PNG preserves these pixels losslessly and supports alpha. JPEG is lossy and does not support an alpha channel. To save transparency as JPEG, composite it onto an explicit background first; simply discarding alpha can produce an unintended background.

This badge has transparent pixels outside the rounded shape. JPEG quality changes compression; it does not increase image resolution.

In [ ]:
badge = overlay.crop((0, 0, 180, 70))
badge.save(output_dir / "pillow_transparent_badge.png")
white_background = Image.new("RGBA", badge.size, "white")
flattened_badge = Image.alpha_composite(white_background, badge).convert("RGB")
flattened_badge.save(output_dir / "pillow_badge.jpg", quality=90)

buffer = io.BytesIO()
loaded.save(buffer, format="PNG")
buffer.seek(0)
with Image.open(buffer) as reopened:
    roundtrip = reopened.convert("RGB")
print("PNG bytes in memory:", len(buffer.getvalue()))
display(flattened_badge)

## 15. Complete exercise: create a contact sheet

Arrange six variants in labeled tiles. Padding preserves each variant's full aspect ratio. The sheet makes it easy to compare transformations in one exported image.

We export PNG, JPEG, and a JSON manifest describing the original and variants. Running the export replaces those named files under `outputs`.

In [ ]:
variants = [("Original", loaded), ("Grayscale", grayscale), ("Crop", cropped),
            ("Blur", blurred), ("Color spotlight", spotlight), ("Overlay", composited)]
tile_width, tile_height = 240, 190
contact_sheet = Image.new("RGB", (tile_width * 3, tile_height * 2), "#edf0f4")
labels = ImageDraw.Draw(contact_sheet)
manifest = []
for index, (label, variant) in enumerate(variants):
    x = (index % 3) * tile_width
    y = (index // 3) * tile_height
    tile = ImageOps.pad(variant.convert("RGB"), (220, 150), method=Image.Resampling.LANCZOS, color="white")
    contact_sheet.paste(tile, (x + 10, y + 10))
    labels.text((x + 10, y + 166), label, font=font, fill="#172033")
    manifest.append({"label": label, "mode": variant.mode, "size": list(variant.size)})
contact_sheet.save(output_dir / "pillow_contact_sheet.png")
contact_sheet.save(output_dir / "pillow_contact_sheet.jpg", quality=90)
(output_dir / "pillow_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print("Saved contact sheet PNG/JPEG and JSON manifest.")
display(contact_sheet)

## 16. Practice and troubleshooting

1. Compare NEAREST and LANCZOS on the same small thumbnail.
2. Move the crop box while keeping it within the source bounds.
3. Change badge opacity and compare the result over light and dark backgrounds.
4. Add a mirrored variant and update the contact-sheet layout for more tiles.

| Symptom | What to check |
| --- | --- |
| JPEG rejects an RGBA image | Composite onto a background and convert to RGB |
| Resized subject looks stretched | Use contain, fit, pad, or thumbnail according to the desired framing |
| Original image changed unexpectedly | Drawing, paste, and thumbnail modify their targets; copy first |
| Array dimensions appear reversed | Pillow size is width/height; NumPy shape is height/width/channels |
| Brightening causes dark pixels | Widen the integer type before arithmetic, then clip |
| A phone photo looks rotated | Apply EXIF orientation before processing |
| Crop includes black areas | Check whether its box extends outside the source |

See the official [Pillow tutorial](https://pillow.readthedocs.io/en/stable/handbook/tutorial.html), [Image reference](https://pillow.readthedocs.io/en/stable/reference/Image.html), and [ImageOps reference](https://pillow.readthedocs.io/en/stable/reference/ImageOps.html).

## Explain what the evidence supports

Explain why your chosen thumbnail operation preserves the content that matters. Reopen one exported image and verify its size, mode and treatment of transparent areas.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: standardize a mixed image upload batch without stretching it

A catalog upload mixes RGB/RGBA modes, wide and tall pictures, transparent backgrounds and one corrupt file. Build fixed-size previews under a pixel budget, quarantine unreadable inputs, and compare stretching with aspect-preserving padding. Images are generated locally to simulate upload cases; there is no private media or network dependency.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through geometry before processing the batch

A 400×200 image has aspect ratio 2. Resizing it directly to 160×160 changes that ratio to 1; circles become ellipses. Padding a 160×80 resized image inside a 160×160 square preserves shape and adds background. Decide whether retaining all content or filling the square matters before choosing crop versus pad.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

from PIL import Image,ImageDraw,ImageOps,UnidentifiedImageError
p2_input=p2_out/'pillow_uploads';p2_input.mkdir(exist_ok=True)
for p2_i,(p2_size,p2_mode) in enumerate([((400,200),'RGB'),((180,420),'RGB'),((240,240),'RGBA')]):
    p2_image=Image.new(p2_mode,p2_size,(255,255,255,0) if p2_mode=='RGBA' else 'white')
    p2_draw=ImageDraw.Draw(p2_image);p2_draw.ellipse((20,20,140,140),fill='tomato')
    p2_image.save(p2_input/f'image_{p2_i}.png')
(p2_input/'broken.png').write_bytes(b'not an image')
p2_source=Image.open(p2_input/'image_0.png')
p2_stretched=p2_source.resize((160,160))
p2_padded=ImageOps.pad(p2_source,(160,160),color='white',method=Image.Resampling.LANCZOS)
p2_sheet=Image.new('RGB',(320,160),'white');p2_sheet.paste(p2_stretched,(0,0));p2_sheet.paste(p2_padded,(160,0))
display(p2_sheet)

### Work through resource limits and failure isolation

Inspect dimensions before decoding all pixels, enforce a 500,000-pixel teaching budget, apply EXIF orientation, and process one image at a time. A compressed file's small byte size does not guarantee a small decoded image. Alpha must be composited onto an explicit background before JPEG export, rather than silently discarded.

Catch image-read errors per file so one bad upload does not erase the batch report. Record accepted and rejected paths, original dimensions and output sizes.

In [ ]:
p2_results=[]
for p2_path in sorted(p2_input.glob('*.png')):
    try:
        with Image.open(p2_path) as p2_image:
            p2_original=p2_image.size
            if p2_image.width*p2_image.height>500000:raise ValueError('pixel budget exceeded')
            p2_rgba=ImageOps.exif_transpose(p2_image).convert('RGBA')
            p2_background=Image.new('RGBA',p2_rgba.size,'white')
            p2_flat=Image.alpha_composite(p2_background,p2_rgba).convert('RGB')
            p2_preview=ImageOps.pad(p2_flat,(160,160),method=Image.Resampling.LANCZOS,color='white')
            p2_destination=p2_out/('pillow_preview_'+p2_path.stem+'.jpg')
            p2_preview.save(p2_destination,quality=85)
            with Image.open(p2_destination) as p2_check:assert p2_check.size==(160,160) and p2_check.mode=='RGB'
            p2_results.append({'file':p2_path.name,'status':'accepted','source_size':str(p2_original),'output_bytes':p2_destination.stat().st_size})
    except (UnidentifiedImageError,OSError,ValueError) as p2_error:
        p2_results.append({'file':p2_path.name,'status':'rejected','reason':str(p2_error)})
p2_report=pd.DataFrame(p2_results);assert (p2_report.status=='accepted').sum()==3 and (p2_report.status=='rejected').sum()==1
p2_report.to_csv(p2_out/'pillow_upload_audit.csv',index=False);display(p2_report)

### Interpret the processing policy

The corruption is isolated and the three previews satisfy the shape/mode contract. Padding preserves all source content but may produce large borders; center-cropping fills the box but can remove the subject. EXIF handling and the white alpha background are explicit decisions. A pixel cap limits this pipeline's input dimensions, not every possible image-decoder resource risk.

**Try:** add a 2,000×2,000 generated image. **Worked reasoning:** it should be rejected before conversion under the same budget. Raising the cap is a capacity decision, not a correctness fix. Compare JPEG quality settings using visible artifacts and output bytes; identical dimensions do not imply identical fidelity.